In [7]:
# Blokada wielowątkowości
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [8]:
# Biblioteki
import numpy as np
from time import perf_counter
import matplotlib.pyplot as plt
from scipy.sparse import diags # Do rzadkiej macierzy
from scipy.sparse.linalg import eigsh

In [9]:
# Klasa 1. do pomiaru czasu.
class Timer():
    def __init__(self, name):
        self.name = name
        self.engine_id = os.environ.get('IPY_ENGINE_ID', '0')
    
    def __enter__(self):
        self.start = perf_counter()
        return self

    def __exit__(self, *args):
        self.end = perf_counter()
        if self.engine_id == '0':
            t = self.end - self.start
            print(f"Time - {self.name}: {t/60:.3f} min")

In [22]:
# Stałe fizyczne.
PI = np.pi
h_bar = 1.0
m = 1.0

In [52]:
# Funkcja 1. do inicjalizacji Hamiltonianu i stałych układu.
def initialize(N, L, D, v_func):
    dx = dy = L / (N-1)
    x = np.zeros(N)
    for i in range(1, N): x[i] = x[i-1] + dx
    if(D==1):
        dtau = 0.1 * m*dx**2 / h_bar
        psi = np.zeros(N)
        v = np.zeros(N)
        for i in range(N):
            v[i] = v_func(x[i], L, D)
            psi[i][j] = x[i]*(L - x[i])
        return N, dtau, psi,  X
    if(D==2):
        dtau = 0.1* m*dx*dy / (2.0*h_bar)
        y = np.zeros(N)
        for i in range(1, N): y[i] = y[i-1]+dy
        psi = np.zeros(N, N)
        v = np.zeros(N, N)
        for i in range(N):
            for j in range(N):
                v[i][j] = v_func(x[i], y[j], L, D)
                psi[i][j] = x[i]*(L - x[i]) + y[i]*(L - y[i])
        return N, dx, dtau, psi, v, x, y

In [56]:
# Funkcje 2. potencjału.
def V_fine(x, y, L, D):
    if(D == 1):
        if(L/3 < x < 2*L/3 and L/3): return 0.0
        return 10.0
    if(D ==2):
        if(L/3 < x < 2*L/3 and L/3 < y < 2*L/3): return 0.0
        return 10.0

In [57]:
# Funkcja 3. energii.
def Energy(psi, v, dx, D):
    e = 0.0
    if(D == 1):
        for i in range(1, N-1):
            e += psi[i](-h_bar**2(psi[i-1] - 2.0*psi[i] + psi[i+1]) / (2.0 * m * dx**2) + v[i]*psi[i])*dx
    if(D == 2):
        for i in range(1, N-1):
                for j in range(1, N-1):
                    e += psi[i][j](-h_bar**2(1000.0) / (2.0 * m * dx**2) + v[i][j]*psi[i][j])*dx

<>:5: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
<>:9: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
<>:5: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
<>:9: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
/tmp/ipykernel_209803/1290822081.py:5: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
  e += psi[i](-h_bar**2(psi[i-1] - 2.0*psi[i] + psi[i+1]) / (2.0 * m * dx**2) + v[i]*psi[i])*dx
/tmp/ipykernel_209803/1290822081.py:9: SyntaxWarning: 'int' object is not callable; perhaps you missed a comma?
  e += psi[i][j](-h_bar**2(1000.0) / (2.0 * m * dx**2) + v[i][j]*psi[i][j])*dx


In [ ]:
# Funkcja 4. normalizacji f. falowej.
def psi_norm(psi, k, x, dx):
    psi[k] = psi[k] / 
    

In [64]:
# Funkcja 3. do metody czasu urojonego.
def imaginary(_N, _L, _D, _v_func):
    _, N, dx, dtau, PSI, V, X, Y = initialize(_N, _L, _D, _v_func)
    PSI_new = np.zeros(N)
    p = 0
    epsilon = 1e-5 #-10
    E_old = Energy(PSI, V, dx, 1)
    norm = np.sqrt(np.sum(PSI**2) * dx)
    PSI = PSI / norm
    E_new = E_old + 1.0
    difference = 1.0
    while(difference > epsilon):
        for i in range(1, N-1):
            PSI_new[i] = h_bar*dtau/(2.0*m*dx**2) * \
                        (PSI[i-1] - 2.0*PSI[i] + PSI[i+1]) - \
                        PSI[i]*(V[i]*dtau / h_bar  - 1.0)
            PSI_new[0] = PSI_new[N] = 0.0
            norm = np.sqrt(np.sum(PSI_new**2) * dx)
            PSI = PSI_new / norm
            E_new = Energy(PSI, V, dx, 1)
            difference = abs(E_new-E_old)
            E_old = E_new
            
    return PSI, E_new # zwraca psi i energię

In [65]:
PSI, E0 = imaginary(100, 10, 1, 0)

NameError: name 'v_func_1' is not defined